In [19]:
!pip install transformers sentencepiece accelerate tqdm -q

In [20]:
import json
import re
import torch

from tqdm import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

from google.colab import files

In [3]:
uploaded = files.upload()

Saving mintaka_dev.json to mintaka_dev.json
Saving mintaka_test.json to mintaka_test.json
Saving mintaka_train.json to mintaka_train.json


In [21]:
with open("mintaka_train.json", "r") as f:
    train_data = json.load(f)

with open("mintaka_dev.json", "r") as f:
    dev_data = json.load(f)

with open("mintaka_test.json", "r") as f:
    test_data = json.load(f)

print("Train Size:", len(train_data))
print("Dev Size  :", len(dev_data))
print("Test Size :", len(test_data))

Train Size: 14000
Dev Size  : 2000
Test Size : 4000


In [22]:
def extract_answer(item):

    ans = item.get("answer", {})

    # structured answer
    if isinstance(ans, dict):

        if "answer" in ans and ans["answer"]:

            obj = ans["answer"][0]

            # dictionary answer
            if isinstance(obj, dict):

                # label field
                if "label" in obj:

                    label = obj["label"]

                    # multilingual label
                    if isinstance(label, dict):

                        if "en" in label:
                            return str(label["en"])

                    return str(label)

                # name field
                if "name" in obj:
                    return str(obj["name"])

            return str(obj)

        # mention fallback
        if "mention" in ans:
            return str(ans["mention"])

    return "UNKNOWN"

In [23]:
def load_mintaka(data):

    questions = []
    answers = []

    for item in data:

        questions.append(
            item["question"]
        )

        answers.append(
            extract_answer(item)
        )

    return questions, answers

train_q, train_a = load_mintaka(train_data)
dev_q, dev_a = load_mintaka(dev_data)
test_q, test_a = load_mintaka(test_data)

print("\nSample Question:")
print(test_q[0])

print("\nSample Answer:")
print(test_a[0])


Sample Question:
What man was a famous American author and also a steamboat pilot on the Mississippi River?

Sample Answer:
Mark Twain


In [40]:
MODEL_NAME = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

print("Using device:", device)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Using device: cuda


In [41]:
def predict_answer(question):

    prompt = f"""
    Answer the following question briefly and accurately.

    Question:
    {question}

    Answer:
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_length=32,
        num_beams=4,
        early_stopping=True
    )

    prediction = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return prediction

In [42]:
preds = []

for q in tqdm(test_q):

    pred = predict_answer(q)

    preds.append(pred)

100%|██████████| 4000/4000 [07:22<00:00,  9.04it/s]


In [35]:
print("\n========== SAMPLE PREDICTIONS ==========\n")

for i in range(10):

    print("QUESTION :", test_q[i])
    print("PREDICTED:", preds[i])
    print("GOLD     :", test_a[i])
    print()


========== SAMPLE PREDICTIONS ==========

QUESTION : What man was a famous American author and also a steamboat pilot on the Mississippi River?
PREDICTED: charles dickens
GOLD     : Mark Twain

QUESTION : How many Academy Awards has Jake Gyllenhaal been nominated for?
PREDICTED: three
GOLD     : 1

QUESTION : Who is older, The Weeknd or Drake?
PREDICTED: The Weeknd
GOLD     : Drake

QUESTION : How many children did Donald Trump have?
PREDICTED: four
GOLD     : 5

QUESTION : Is the main hero in Final Fantasy IX named Kuja?
PREDICTED: no
GOLD     : False

QUESTION : Who performed at the Super Bowl XXIII halftime show?
PREDICTED: michaelangelo michaelangelo
GOLD     : Elvis Presto

QUESTION : Did Free Guy come out in 2021?
PREDICTED: Free Guy came out in 1921.
GOLD     : True

QUESTION : How many countries were in the Central Powers alliance in World War I?
PREDICTED: seven
GOLD     : 4

QUESTION : When was the first Donkey Kong arcade game released?
PREDICTED: 1984
GOLD     : 1981

QUES

In [36]:
def normalize(text):

    text = str(text).lower()

    text = re.sub(r"[^a-z0-9 ]", "", text)

    text = " ".join(text.split())

    return text

In [37]:
def f1_score(pred, gold):

    pred_tokens = normalize(pred).split()

    gold_tokens = normalize(gold).split()

    common = set(pred_tokens) & set(gold_tokens)

    if len(common) == 0:
        return 0.0

    precision = len(common) / len(pred_tokens)

    recall = len(common) / len(gold_tokens)

    return (
        2 * precision * recall
    ) / (precision + recall)

In [38]:
hit1 = 0
f1_total = 0

for pred, gold in zip(preds, test_a):

    # exact match
    if normalize(pred) == normalize(gold):
        hit1 += 1

    # f1
    f1_total += f1_score(
        pred,
        gold
    )

hit1 = hit1 / len(test_a)

f1 = f1_total / len(test_a)

# simplified metrics
hit5 = hit1
mrr = hit1
accuracy = hit1

In [39]:
print("\n========== FLAN-T5 ZERO-SHOT RESULTS ==========\n")

print("Hit@1    :", hit1)
print("Hit@5    :", hit5)
print("MRR      :", mrr)
print("F1 Score :", f1)
print("Accuracy :", accuracy)


========== FLAN-T5 ZERO-SHOT RESULTS ==========

Hit@1    : 0.04025
Hit@5    : 0.04025
MRR      : 0.04025
F1 Score : 0.09585039094892044
Accuracy : 0.04025
